# 09 - Pandas: fundamentos y primeras transformaciones

En este notebook aprenderás a:

- Crear y cargar **DataFrames** y **Series**.
- Explorar datos: `.head()`, `.info()`, `.describe()`.
- Seleccionar columnas y filas (`[]`, `.loc`, `.iloc`).
- Filtrar, ordenar, crear y transformar columnas.
- Agrupar con `groupby` y obtener estadísticas.
- Guardar y cargar a CSV/Parquet.


## 1. Importación y dataset sintético

In [3]:
# Creación de una series
import pandas as pd
s = pd.Series([10, 20, 30], name="Ventas")
s, s.dtype, s.shape, s.size, s.ndim

(0    10
 1    20
 2    30
 Name: Ventas, dtype: int64,
 dtype('int64'),
 (3,),
 3,
 1)

In [8]:
df = pd.DataFrame({"Producto": ["A", "B", "C"], "Ventas": [100, 250, 300], "Precio": [10.5, 12.3, 9.8] })
df, df.dtypes, df.shape, df.size, df.ndim

(  Producto  Ventas  Precio
 0        A     100    10.5
 1        B     250    12.3
 2        C     300     9.8,
 Producto     object
 Ventas        int64
 Precio      float64
 dtype: object,
 (3, 3),
 9,
 2)

In [31]:
import pandas as pd
import numpy as np

print("Pandas versión:", pd.__version__)

rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    "id": range(1, n+1),
    "edad": rng.integers(18, 70, size=n),
    "ingresos": rng.normal(25000, 8000, size=n).round(2),
    "ciudad": rng.choice(["Cádiz", "Sevilla", "Málaga", "Córdoba"], size=n),
    "compra": rng.choice([0,1], size=n, p=[0.4, 0.6]),
})
df.head()

Pandas versión: 2.3.3


,id,edad,ingresos,ciudad,compra
0,1,22,28198.19,Sevilla,1
1,2,58,17756.17,Sevilla,1
2,3,52,21974.70,Cádiz,0
3,4,40,35393.83,Sevilla,1
4,5,40,22149.89,Málaga,0


In [11]:
print(df.head(10))
print(df.tail())
print(df.sample(5))
print(df.info()) # Información general del DataFrame
# Estadísticas descriptivas para todas las columnas, sin importar el tipo de dato
# si no ponemos include='all', solo se muestran las columnas numéricas por defecto
print(df.describe(include='all'))
# Estadísticas descriptivas para columnas numéricas ecluyendo la columna 'id'
print(df.describe(include=[np.number]).drop(columns=['id']))

   id  edad  ingresos   ciudad  compra
0   1    22  28198.19  Sevilla       1
1   2    58  17756.17  Sevilla       1
2   3    52  21974.70    Cádiz       0
3   4    40  35393.83  Sevilla       1
4   5    40  22149.89   Málaga       0
5   6    62  30900.12    Cádiz       0
6   7    22  17531.06  Sevilla       0
7   8    54  23356.50  Sevilla       1
8   9    28  17399.82   Málaga       0
9  10    22  22287.74  Sevilla       1
      id  edad  ingresos   ciudad  compra
195  196    22   4466.73  Córdoba       1
196  197    35  23105.20    Cádiz       1
197  198    24  26412.10    Cádiz       0
198  199    35  27367.95   Málaga       1
199  200    68  22024.68  Córdoba       1
      id  edad  ingresos   ciudad  compra
159  160    58  21990.75  Córdoba       1
2      3    52  21974.70    Cádiz       0
97    98    53  17842.18  Sevilla       1
185  186    41  28635.36  Córdoba       0
120  121    40  23763.95  Córdoba       1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to

## 2. Exploración rápida

In [12]:
df.shape, df.columns, df.dtypes

((200, 5),
 Index(['id', 'edad', 'ingresos', 'ciudad', 'compra'], dtype='object'),
 id            int64
 edad          int64
 ingresos    float64
 ciudad       object
 compra        int64
 dtype: object)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        200 non-null    int64  
 1   edad      200 non-null    int64  
 2   ingresos  200 non-null    float64
 3   ciudad    200 non-null    object 
 4   compra    200 non-null    int64  
dtypes: float64(1), int64(3), object(1)
memory usage: 7.9+ KB


In [15]:
df.describe(include=[np.number])

,id,edad,ingresos,compra
count,200.000000,200.000000,200.000000,200.000000
mean,100.500000,43.955000,24745.052450,0.615000
std,57.879185,14.286831,7957.136843,0.487816
min,1.000000,18.000000,4466.730000,0.000000
25%,50.750000,32.000000,19190.475000,0.000000
50%,100.500000,44.000000,24111.490000,1.000000
75%,150.250000,57.000000,28476.075000,1.000000
max,200.000000,69.000000,48310.900000,1.000000


## 3. Selección de columnas y filas

In [16]:
# Selección por nombre
edades = df["edad"]
subset = df[["edad", "ingresos", "compra"]]
edades.head(), subset.head()

(0    22
 1    58
 2    52
 3    40
 4    40
 Name: edad, dtype: int64,
    edad  ingresos  compra
 0    22  28198.19       1
 1    58  17756.17       1
 2    52  21974.70       0
 3    40  35393.83       1
 4    40  22149.89       0)

In [20]:
# Filtro por condición
mayores_50 = df[df["edad"] > 50]
mayores_50.head()

,id,edad,ingresos,ciudad,compra
1,2,58,17756.17,Sevilla,1
2,3,52,21974.70,Cádiz,0
5,6,62,30900.12,Cádiz,0
7,8,54,23356.50,Sevilla,1
11,12,68,11181.44,Sevilla,0


In [26]:
# loc vs iloc
primera_fila = df.loc[0]
fila_10 = df.iloc[9]

#accediendo por número de fila y nombre de columna
valor = df.loc[4, "ingresos"]

# Accediendo por número de fila y número de columna
valor2 = df.iloc[4, 2]
print(valor, valor2)
primera_fila, fila_10

22149.89 22149.89


(id                 1
 edad              22
 ingresos    28198.19
 ciudad       Sevilla
 compra             1
 Name: 0, dtype: object,
 id                10
 edad              22
 ingresos    22287.74
 ciudad       Sevilla
 compra             1
 Name: 9, dtype: object)

## 4. Ordenación, columnas nuevas y transformaciones

In [32]:
df_sorted = df.sort_values(["ciudad", "ingresos"], ascending=[True, False])
print(df_sorted.head())
print(df.head())
# hacemos el ordenamiento en el mismo DataFrame
print("Ordenando el DataFrame por ingresos de forma descendente...")
df.sort_values("ingresos", ascending=False, inplace=True)
print(df.head())

      id  edad  ingresos ciudad  compra
64    65    39  42027.76  Cádiz       0
40    41    26  40973.85  Cádiz       0
129  130    51  40014.76  Cádiz       1
57    58    26  39143.44  Cádiz       1
78    79    47  37632.73  Cádiz       1
   id  edad  ingresos   ciudad  compra
0   1    22  28198.19  Sevilla       1
1   2    58  17756.17  Sevilla       1
2   3    52  21974.70    Cádiz       0
3   4    40  35393.83  Sevilla       1
4   5    40  22149.89   Málaga       0
Ordenando el DataFrame por ingresos de forma descendente...
      id  edad  ingresos   ciudad  compra
41    42    57  48310.90   Málaga       1
150  151    62  48240.54   Málaga       0
128  129    47  45139.79  Sevilla       0
146  147    19  42033.98   Málaga       0
64    65    39  42027.76    Cádiz       0


In [33]:
# Columna nueva: tramo_edad
#cut para crear intervalos
df = df.assign(
    tramo_edad = pd.cut(df["edad"], bins=[17,25,35,50,70], labels=["18-25","26-35","36-50","51-70"])
)
df.head()

,id,edad,ingresos,ciudad,compra,tramo_edad
41,42,57,48310.90,Málaga,1,51-70
150,151,62,48240.54,Málaga,0,51-70
128,129,47,45139.79,Sevilla,0,36-50
146,147,19,42033.98,Málaga,0,18-25
64,65,39,42027.76,Cádiz,0,36-50


In [34]:
# Otras formas de añadir columnas
df["ingresos_anuales"] = df["ingresos"] * 12
df.head()

,id,edad,ingresos,ciudad,compra,tramo_edad,ingresos_anuales
41,42,57,48310.90,Málaga,1,51-70,579730.80
150,151,62,48240.54,Málaga,0,51-70,578886.48
128,129,47,45139.79,Sevilla,0,36-50,541677.48
146,147,19,42033.98,Málaga,0,18-25,504407.76
64,65,39,42027.76,Cádiz,0,36-50,504333.12


In [37]:
# Transformación con apply (evitar si hay alternativa vectorizada)
def etiqueta_ingresos(x):
    if x < 20000: return "bajo"
    elif x < 30000: return "medio"
    else: return "alto"

df["nivel_ingresos"] = df["ingresos"].apply(etiqueta_ingresos)
df.tail(10)

,id,edad,ingresos,ciudad,compra,tramo_edad,ingresos_anuales,nivel_ingresos
145,146,62,13263.64,Córdoba,0,51-70,159163.68,bajo
36,37,62,13233.55,Córdoba,1,51-70,158802.60,bajo
54,55,43,11602.54,Málaga,1,36-50,139230.48,bajo
11,12,68,11181.44,Sevilla,0,51-70,134177.28,bajo
183,184,54,11043.31,Córdoba,1,51-70,132519.72,bajo
144,145,69,10938.17,Sevilla,0,51-70,131258.04,bajo
106,107,58,8598.62,Málaga,0,51-70,103183.44,bajo
44,45,21,7943.63,Málaga,0,18-25,95323.56,bajo
138,139,36,7821.69,Cádiz,1,36-50,93860.28,bajo
195,196,22,4466.73,Córdoba,1,18-25,53600.76,bajo


## 5. Agrupaciones y estadísticas

In [ ]:
agg_por_ciudad = (
    df.groupby("ciudad")
      .agg(media_ingresos=("ingresos","mean"),
           tasa_compra=("compra","mean"),
           n=("id","count"))
      .reset_index()
)
agg_por_ciudad

## 6. Guardar y cargar datos (CSV/Parquet)

In [ ]:
# Guardar
csv_path = "/mnt/data/clientes_sintetico.csv"
parquet_path = "/mnt/data/clientes_sintetico.parquet"
df.to_csv(csv_path, index=False)
try:
    df.to_parquet(parquet_path, index=False)
except Exception as e:
    print("Parquet opcional (puede requerir pyarrow/fastparquet). Error:", e)

csv_path, parquet_path

In [ ]:
# Cargar desde CSV
df2 = pd.read_csv("/mnt/data/clientes_sintetico.csv")
df2.head()

## 📝 Ejercicios

1) Crea una columna `gasto_relativo = ingresos / ingresos.mean()`.  
2) Calcula la tasa de compra por `tramo_edad` y ordénala de mayor a menor.  
3) Filtra los clientes de **Sevilla** con `ingresos > 30000` y guarda el resultado en CSV.
